# Offline Language Translation System (LTS)

### Project Track: Natural Language Processing (NLP) / LLM-based System

## 1. Problem Definition & Objective

**Problem Statement:**
Most modern language translation systems rely on continuous internet connectivity and cloud-based APIs. This limits usability in low-connectivity environments and raises privacy concerns.

**Objective:**
- Build a fully offline languageC-based language translation system
- Support multiple Indian and international languages
- Provide CLI and GUI interfaces
- Ensure privacy-preserving, standalone deployment

**Real-world Relevance:**
- Rural and low-connectivity areas
- Privacy-sensitive translations
- Educational and local language accessibility

## 2. Data Understanding & Preparation

**Dataset Source:**
- Public multilingual datasets used during pretraining of IndicTrans2 models
- No custom dataset collection required (model-based translation)

**Data Nature:**
- User-provided text input
- Real-time inference

**Preprocessing Steps:**
- Language normalization
- Tokenization using pretrained tokenizer
- Indic text normalization using IndicProcessor

**Missing Values / Noise Handling:**
- Not applicable (direct user input text)

## 3. Model / System Design

**AI Technique Used:**
- NLP with Transformer-based Large Language Models (LLMs)

**Models Used:**
- Indic → English: indictrans2-indic-en-dist-200M
- English → Indic: indictrans2-en-indic-dist-200M

**Architecture:**
Input Text → Preprocessing → Tokenization → Offline Model Inference → Postprocessing → Output Text

**Design Justification:**
- Distilled 200M models for reduced memory usage
- Offline inference for privacy and availability
- English pivoting for Indic-to-Indic translation

## 4. Core Implementation

In [1]:
# Import required libraries
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

/home/snehal-modgil/anaconda3/envs/offline-lts/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 4.1 Offline Translation Engine

In [2]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

# Constants for offline models
# 200M models are faster and require less RAM for offline use
INDIC_EN_MODEL = "ai4bharat/indictrans2-indic-en-dist-200M"
EN_INDIC_MODEL = "ai4bharat/indictrans2-en-indic-dist-200M"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class OfflineTranslator:
    def __init__(self):
        print(f"Loading models on {DEVICE}...")
        
        # Initialize Processor
        self.ip = IndicProcessor(inference=True)
        
        # Load Indic -> English Model
        self.tokenizer_indic_en = AutoTokenizer.from_pretrained(INDIC_EN_MODEL, trust_remote_code=True)
        self.model_indic_en = AutoModelForSeq2SeqLM.from_pretrained(INDIC_EN_MODEL, trust_remote_code=True).to(DEVICE)
        
        # Load English -> Indic Model
        self.tokenizer_en_indic = AutoTokenizer.from_pretrained(EN_INDIC_MODEL, trust_remote_code=True)
        self.model_en_indic = AutoModelForSeq2SeqLM.from_pretrained(EN_INDIC_MODEL, trust_remote_code=True).to(DEVICE)

        self.model_indic_en.eval()
        self.model_en_indic.eval()
        
        self.lang_codes = {
            "english": "eng_Latn",
            "hindi": "hin_Deva",
            "tamil": "tam_Taml",
            "telugu": "tel_Telu",
            "bengali": "ben_Beng",
            "marathi": "mar_Deva",
            "gujarati": "guj_Gujr",
            "punjabi": "pan_Guru",
            "kannada": "kan_Knda",
            "malayalam": "mal_Mlym",
            "odia": "ory_Orya",
            "assamese": "asm_Beng",
            "urdu": "urd_Arab",
            "nepali": "npi_Deva",
            "sanskrit": "san_Deva",
            "maithili": "mai_Deva"
        }

        self.supported_langs = set(self.lang_codes.values())  

    def get_supported_languages(self):
        """
        Returns a dict: {Language Name: Language Code}
        """
        return self.lang_codes

    def _run_inference(self, text, src, tgt, model, tokenizer):
        # 1. Preprocess: This formats the string to include the correct tags
        # and avoids the "Invalid source language tag" error
        batch = self.ip.preprocess_batch([text], src_lang=src, tgt_lang=tgt)
        
        # 2. Tokenize
        inputs = tokenizer(
            batch, 
            return_tensors="pt", 
            padding=True, 
            truncation=True
        ).to(DEVICE)
        
        # 3. Generate
        with torch.no_grad():
            generated_tokens = model.generate(
                **inputs, 
                max_length=256, 
                num_beams=5, 
                use_cache=False
            )
        
        # 4. Postprocess: Decodes and cleans the output
        decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        return self.ip.postprocess_batch(decoded, lang=tgt)[0]
    
    def _normalize_lang(self, lang):
        # If already a code, return as-is
        if lang in self.supported_langs:
            return lang

        # If it's a name, convert to code
        if lang in self.lang_codes:
            return self.lang_codes[lang]

        raise ValueError(f"Unsupported language: {lang}")


    def translate(self, text, src_lang, tgt_lang):
        # Validate language codes (case-sensitive)
        src_lang = self._normalize_lang(src_lang)
        tgt_lang = self._normalize_lang(tgt_lang)

        if src_lang not in self.supported_langs:
            raise ValueError(f"Unsupported source language: {src_lang}")
    
        if tgt_lang not in self.supported_langs:
            raise ValueError(f"Unsupported target language: {tgt_lang}")
    
        # Case 1: Indic -> English
        if tgt_lang == "eng_Latn" and src_lang != "eng_Latn":
            return self._run_inference(
                text,
                src_lang,
                "eng_Latn",
                self.model_indic_en,
                self.tokenizer_indic_en
            )
    
        # Case 2: English -> Indic
        if src_lang == "eng_Latn" and tgt_lang != "eng_Latn":
            return self._run_inference(
                text,
                "eng_Latn",
                tgt_lang,
                self.model_en_indic,
                self.tokenizer_en_indic
            )
    
        # Case 3: Indic -> Indic (pivot through English)
        if src_lang != "eng_Latn" and tgt_lang != "eng_Latn":
            # Step A: Indic -> English
            intermediate_en = self._run_inference(
                text,
                src_lang,
                "eng_Latn",
                self.model_indic_en,
                self.tokenizer_indic_en
            )
    
            # Step B: English -> Indic
            return self._run_inference(
                intermediate_en,
                "eng_Latn",
                tgt_lang,
                self.model_en_indic,
                self.tokenizer_en_indic
            )
    
        # Case 4: Same source and target (should not happen via CLI)
        return text


if __name__ == "__main__":
    translator = OfflineTranslator()
    
    # Test 1: Hindi to English
    # print("Hindi -> English:", translator.translate("आप कैसे हैं?", "hindi", "english"))
    
    # Test 2: Hindi to Tamil (Pivoting automatically)
    # print("Hindi -> Tamil:", translator.translate("नमस्ते, आप कैसे हैं?", "hindi", "tamil"))

Loading models on cuda...


### 4.2 Command Line Interface (CLI)

In [ ]:
LANGUAGES = {
    "English": "eng_Latn",
    "Hindi": "hin_Deva",
    "Tamil": "tam_Taml",
    "Telugu": "tel_Telu",
    "Bengali": "ben_Beng",
    "Marathi": "mar_Deva",
    "Gujarati": "guj_Gujr",
    "Punjabi": "pan_Guru",
    "Kannada": "kan_Knda",
    "Malayalam": "mal_Mlym",
    "Odia": "ory_Orya",
    "Assamese": "asm_Beng",
    "Urdu": "urd_Arab",
    "Nepali": "npi_Deva",
    "Sanskrit": "san_Deva",
    "Maithili": "mai_Deva"
}

def show_languages():
    print("\nSupported Languages:")
    for idx, lang in enumerate(LANGUAGES.keys(), start=1):
        print(f"{idx}. {lang}")

def get_language(prompt):
    while True:
        show_languages()
        choice = input(prompt).strip()

        # Numeric selection
        if choice.isdigit():
            idx = int(choice) - 1
            if 0 <= idx < len(LANGUAGES):
                lang_name = list(LANGUAGES.keys())[idx]
                return lang_name, LANGUAGES[lang_name]

        # Text selection
        for name in LANGUAGES:
            if choice.lower() == name.lower():
                return name, LANGUAGES[name]

        print("❌ Invalid choice. Try again.")


def main():
    print("\n=== Offline Indian Language Translation System ===")

    translator = OfflineTranslator()

    while True:
        src_name, src_code = get_language("\nSelect SOURCE language: ")
        tgt_name, tgt_code = get_language("Select TARGET language: ")

        if src_code == tgt_code:
            print("❌ Source and target languages cannot be the same.")
            continue

        text = input(f"\nEnter text in {src_name} (or type 'exit'): ").strip()
        if text.lower() == "exit":
            print("Exiting translator.")
            break

        try:
            output = translator.translate(text, src_code, tgt_code)
            print(f"\nTranslated ({tgt_name}):")
            print(output)
        except Exception as e:
            print("❌ Translation failed:", e)

if __name__ == "__main__":
    main()


=== Offline Indian Language Translation System ===
Loading models on cuda...

Supported Languages:
1. English
2. Hindi
3. Tamil
4. Telugu
5. Bengali
6. Marathi
7. Gujarati
8. Punjabi
9. Kannada
10. Malayalam
11. Odia
12. Assamese
13. Urdu
14. Nepali
15. Sanskrit
16. Maithili

Supported Languages:
1. English
2. Hindi
3. Tamil
4. Telugu
5. Bengali
6. Marathi
7. Gujarati
8. Punjabi
9. Kannada
10. Malayalam
11. Odia
12. Assamese
13. Urdu
14. Nepali
15. Sanskrit
16. Maithili

Translated (Hindi):
आप कैसे हैं?

Supported Languages:
1. English
2. Hindi
3. Tamil
4. Telugu
5. Bengali
6. Marathi
7. Gujarati
8. Punjabi
9. Kannada
10. Malayalam
11. Odia
12. Assamese
13. Urdu
14. Nepali
15. Sanskrit
16. Maithili
❌ Invalid choice. Try again.

Supported Languages:
1. English
2. Hindi
3. Tamil
4. Telugu
5. Bengali
6. Marathi
7. Gujarati
8. Punjabi
9. Kannada
10. Malayalam
11. Odia
12. Assamese
13. Urdu
14. Nepali
15. Sanskrit
16. Maithili


KeyboardInterrupt: Interrupted by user

### 4.3 Graphical User Interface (GUI)

In [3]:
import tkinter as tk
from tkinter import ttk, messagebox

LANGUAGES = {
    "English": "eng_Latn",
    "Hindi": "hin_Deva",
    "Tamil": "tam_Taml",
    "Telugu": "tel_Telu",
    "Bengali": "ben_Beng",
    "Marathi": "mar_Deva",
    "Gujarati": "guj_Gujr",
    "Punjabi": "pan_Guru",
    "Kannada": "kan_Knda",
    "Malayalam": "mal_Mlym",
    "Odia": "ory_Orya",
    "Assamese": "asm_Beng",
    "Urdu": "urd_Arab",
    "Nepali": "npi_Deva",
    "Sanskrit": "san_Deva",
    "Maithili": "mai_Deva"
}

class TranslatorGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Offline Indian Language Translator")
        self.root.geometry("1000x520")
        self.root.configure(bg="#f2f4f7")

        self.translator = None  # lazy loading

        # ===== TITLE =====
        tk.Label(
            root,
            text="Offline Indian Language Translation System",
            font=("Segoe UI", 18, "bold"),
            bg="#f2f4f7"
        ).pack(pady=10)

        # ===== MAIN FRAME =====
        main_frame = tk.Frame(root, bg="#f2f4f7")
        main_frame.pack(padx=20, pady=10, fill="both", expand=True)

        # ===== LEFT (INPUT) =====
        left_frame = tk.Frame(main_frame, bg="white", bd=0)
        left_frame.grid(row=0, column=0, padx=10, sticky="nsew")

        tk.Label(left_frame, text="Source Language",
                 font=("Segoe UI", 10, "bold"),
                 bg="white").pack(pady=(10, 5))

        self.src_lang = ttk.Combobox(
            left_frame,
            values=list(LANGUAGES.keys()),
            state="readonly",
            width=25
        )
        self.src_lang.current(0)
        self.src_lang.pack(pady=5)

        self.input_text = tk.Text(
            left_frame,
            height=14,
            width=45,
            font=("Segoe UI", 11),
            bd=1,
            relief="solid"
        )
        self.input_text.pack(padx=10, pady=10)

        # ===== RIGHT (OUTPUT) =====
        right_frame = tk.Frame(main_frame, bg="white", bd=0)
        right_frame.grid(row=0, column=1, padx=10, sticky="nsew")

        tk.Label(right_frame, text="Target Language",
                 font=("Segoe UI", 10, "bold"),
                 bg="white").pack(pady=(10, 5))

        self.tgt_lang = ttk.Combobox(
            right_frame,
            values=list(LANGUAGES.keys()),
            state="readonly",
            width=25
        )
        self.tgt_lang.current(1)
        self.tgt_lang.pack(pady=5)

        self.output_text = tk.Text(
            right_frame,
            height=14,
            width=45,
            font=("Segoe UI", 11),
            bd=1,
            relief="solid",
            state="disabled"
        )
        self.output_text.pack(padx=10, pady=10)

        # ===== BUTTONS =====
        button_frame = tk.Frame(root, bg="#f2f4f7")
        button_frame.pack(pady=10)

        tk.Button(
            button_frame,
            text="Translate",
            font=("Segoe UI", 12, "bold"),
            bg="#4CAF50",
            fg="white",
            relief="flat",
            padx=25,
            pady=8,
            command=self.translate_text
        ).grid(row=0, column=0, padx=10)

        tk.Button(
            button_frame,
            text="Copy Output",
            font=("Segoe UI", 11),
            bg="#2196F3",
            fg="white",
            relief="flat",
            padx=20,
            pady=8,
            command=self.copy_output
        ).grid(row=0, column=1, padx=10)

        main_frame.columnconfigure(0, weight=1)
        main_frame.columnconfigure(1, weight=1)

    def translate_text(self):
        text = self.input_text.get("1.0", tk.END).strip()
        if not text:
            messagebox.showerror("Error", "Please enter text to translate.")
            return

        src = LANGUAGES[self.src_lang.get()]
        tgt = LANGUAGES[self.tgt_lang.get()]

        if src == tgt:
            messagebox.showerror("Error", "Source and target languages cannot be the same.")
            return

        try:
            if self.translator is None:
                messagebox.showinfo(
                    "Loading Models",
                    "Loading translation models for the first time.\nPlease wait..."
                )
                self.translator = OfflineTranslator()

            result = self.translator.translate(text, src, tgt)

            self.output_text.config(state="normal")
            self.output_text.delete("1.0", tk.END)
            self.output_text.insert(tk.END, result)
            self.output_text.config(state="disabled")

        except Exception as e:
            messagebox.showerror("Translation Error", str(e))

    def copy_output(self):
        self.root.clipboard_clear()
        text = self.output_text.get("1.0", tk.END).strip()
        if text:
            self.root.clipboard_append(text)
            messagebox.showinfo("Copied", "Translated text copied to clipboard.")

if __name__ == "__main__":
    root = tk.Tk()
    app = TranslatorGUI(root)
    root.mainloop()


Loading models on cuda...


KeyboardInterrupt: 

## 5. Evaluation & Analysis

**Evaluation Method:**
- Qualitative evaluation of translation accuracy
- Performance observation (response time)

**Sample Output:**
- Hindi → English
- English → Tamil

**Limitations:**
- Grammar errors in complex sentences
- Large model size for very low-end systems

## 6. Ethical Considerations & Responsible AI

- Possible bias inherited from training data
- Unequal language representation
- No user data is stored or transmitted
- Offline inference ensures privacy
- Responsible usage disclaimer applied

## 7. Conclusion & Future Scope

**Conclusion:**
The Offline LTS successfully enables multilingual text translation without internet dependency while maintaining acceptable accuracy and performance.

**Future Scope:**
- Add more Indian and global languages
- Speech-to-text and text-to-speech integration
- Mobile and embedded deployment
- Model fine-tuning for domain-specific accuracy

In [ ]:
translator = OfflineTranslator()
translator.translate("how are you", "english", "hindi")

Loading models on cuda...


AttributeError: 'NoneType' object has no attribute 'shape'